In [43]:
!pip install pandas
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd
df = pd.read_csv(r"cleaned_retail.csv")



In [6]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Discount,PaymentMethod,...,Category,SalesChannel,ReturnStatus,ShipmentProvider,WarehouseLocation,OrderPriority,TotalRevenue,SaleDate,Month,Year
0,221958,SKU_1964,White Mug,38,2020-01-01 00:00:00,1.71,37039.0,Australia,0.47,Bank Transfer,...,Apparel,In-store,Not Returned,UPS,London,Medium,34.4394,2020-01-01,1,2020
1,771155,SKU_1241,White Mug,18,2020-01-01 01:00:00,41.25,19144.0,Spain,0.19,PayPal,...,Electronics,Online,Not Returned,UPS,Rome,Medium,601.4250,2020-01-01,1,2020
2,231932,SKU_1501,Headphones,49,2020-01-01 02:00:00,29.11,50472.0,Germany,0.35,Bank Transfer,...,Electronics,Online,Returned,UPS,Berlin,High,927.1535,2020-01-01,1,2020
3,465838,SKU_1760,Desk Lamp,14,2020-01-01 03:00:00,76.68,96586.0,Netherlands,0.14,PayPal,...,Accessories,Online,Not Returned,Royal Mail,Rome,Low,923.2272,2020-01-01,1,2020
4,744167,SKU_1006,Office Chair,47,2020-01-01 05:00:00,70.16,53887.0,Sweden,0.48,Credit Card,...,Electronics,Online,Not Returned,DHL,London,Medium,1714.7104,2020-01-01,1,2020


In [18]:
df.isna().sum()

InvoiceNo               0
StockCode               0
Description             0
Quantity                0
InvoiceDate             0
UnitPrice               0
CustomerID           2489
Country                 0
Discount                0
PaymentMethod           0
ShippingCost            0
Category                0
SalesChannel            0
ReturnStatus            0
ShipmentProvider        0
WarehouseLocation       0
OrderPriority           0
TotalRevenue            0
SaleDate                0
Month                   0
Year                    0
dtype: int64

In [19]:
df=df.dropna(subset=['CustomerID'])

In [20]:
df.isna().sum()

InvoiceNo            0
StockCode            0
Description          0
Quantity             0
InvoiceDate          0
UnitPrice            0
CustomerID           0
Country              0
Discount             0
PaymentMethod        0
ShippingCost         0
Category             0
SalesChannel         0
ReturnStatus         0
ShipmentProvider     0
WarehouseLocation    0
OrderPriority        0
TotalRevenue         0
SaleDate             0
Month                0
Year                 0
dtype: int64

In [21]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Discount,PaymentMethod,...,Category,SalesChannel,ReturnStatus,ShipmentProvider,WarehouseLocation,OrderPriority,TotalRevenue,SaleDate,Month,Year
0,221958,SKU_1964,White Mug,38,2020-01-01 00:00:00,1.71,37039.0,Australia,0.47,Bank Transfer,...,Apparel,In-store,Not Returned,UPS,London,Medium,34.4394,2020-01-01,1,2020
1,771155,SKU_1241,White Mug,18,2020-01-01 01:00:00,41.25,19144.0,Spain,0.19,PayPal,...,Electronics,Online,Not Returned,UPS,Rome,Medium,601.4250,2020-01-01,1,2020
2,231932,SKU_1501,Headphones,49,2020-01-01 02:00:00,29.11,50472.0,Germany,0.35,Bank Transfer,...,Electronics,Online,Returned,UPS,Berlin,High,927.1535,2020-01-01,1,2020
3,465838,SKU_1760,Desk Lamp,14,2020-01-01 03:00:00,76.68,96586.0,Netherlands,0.14,PayPal,...,Accessories,Online,Not Returned,Royal Mail,Rome,Low,923.2272,2020-01-01,1,2020
4,744167,SKU_1006,Office Chair,47,2020-01-01 05:00:00,70.16,53887.0,Sweden,0.48,Credit Card,...,Electronics,Online,Not Returned,DHL,London,Medium,1714.7104,2020-01-01,1,2020


In [48]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
last_purchase=df['InvoiceDate'].max()
rfm=df.groupby('CustomerID').agg(
    rec=('InvoiceDate',lambda x:(last_purchase-x.max()).days),
    freq=('InvoiceDate','count'),
    mtry=("TotalRevenue",'sum')).reset_index()
rfm.head(10)


,CustomerID,rec,freq,mtry
0,10001.0,741,1,1470.7334
1,10003.0,1841,1,365.7385
2,10005.0,250,2,2350.1430
3,10008.0,144,1,48.8280
4,10009.0,1825,1,463.5904
5,10010.0,692,2,2660.4084
6,10011.0,852,1,634.5930
7,10012.0,1435,1,444.3660
8,10017.0,91,1,2729.7360
9,10018.0,654,1,1204.9632


In [41]:
sc=StandardScaler()
rfm_scaled=sc.fit_transform(rfm[['rec','freq','mtry']])
kmeans = KMeans(n_clusters=6, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
rfm['Segment'] = rfm['Cluster'].map({
    0:'Loyal Customer',
    1:'Lost Customer',
    2:'Potential Customer',
    3:'Occasional Buyer',
    4:'At Risk',
    5:'Big Spender',        
       
               
      
        
})
rfm.head(10)

,CustomerID,rec,freq,mtry,Cluster,Segment
0,10001.0,741,1,1470.7334,3,Occasional Buyer
1,10003.0,1841,1,365.7385,1,Lost Customer
2,10005.0,250,2,2350.1430,5,Big Spender
3,10008.0,144,1,48.8280,3,Occasional Buyer
4,10009.0,1825,1,463.5904,1,Lost Customer
5,10010.0,692,2,2660.4084,2,Potential Customer
6,10011.0,852,1,634.5930,4,At Risk
7,10012.0,1435,1,444.3660,1,Lost Customer
8,10017.0,91,1,2729.7360,0,Loyal Customer
9,10018.0,654,1,1204.9632,3,Occasional Buyer


In [42]:
rfm.to_csv(r"new_dataset.csv",index=False)